# 06 - Venue locations
Maps each venue to specific location coordinates
Produces geocoding responses stored in the bucket (one compressed file), and the tables tennis_clean.venue_name_map and tennis_clean.venue_locations
Run after 05_match_dates

## Install the pinned libraries

In [ ]:

# Read from a git tag rather than from main, so a later re-run installs exactly the same versions.
CODE_VERSION = "code-v1"      # the git tag of the code being run: create it on GitHub before running
REPO = "katiedevane/tennis-project"
REQUIREMENTS_URL = f"https://raw.githubusercontent.com/{REPO}/{CODE_VERSION}/requirements.txt"
!pip install -q -r $REQUIREMENTS_URL

# Shared provenance helpers (code version, environment, index of data versions), from the same tag
PROVENANCE_URL = f"https://raw.githubusercontent.com/{REPO}/{CODE_VERSION}/provenance.py"
!wget -q -O provenance.py $PROVENANCE_URL

## Sign in and load settings from Colab Secrets
Switch on Notebook access for both secrets in this notebook.

In [ ]:
import os
from google.colab import auth, userdata

auth.authenticate_user()

PROJECT_ID = userdata.get("TENNIS_PROJECT_ID")
BUCKET_NAME = userdata.get("TENNIS_BUCKET_NAME")
REGION = "EU"
DATASETS = ["tennis_raw", "tennis_clean", "tennis_features"]
os.environ["CLOUDSDK_CORE_PROJECT"] = PROJECT_ID

assert PROJECT_ID and BUCKET_NAME, "A secret is empty: check its name and Notebook access"
print("Settings loaded from Colab Secrets.")

## Record the code version and set the random seed
The git tag, commit, Python environment and random seed of this run are added to every manifest this notebook writes (see provenance.py).
`SEED` is the one project-wide seed: `start_run` applies it, and it should be passed wherever a function takes a seed (e.g. `random_state=SEED`).

In [ ]:
from provenance import start_run, register, table_version, sha256, utc_stamp, SEED

NOTEBOOK = "06_venues"
RUN_INFO = start_run(f"https://github.com/{REPO}", CODE_VERSION, NOTEBOOK)

## Helpers

In [ ]:
import pandas as pd
from google.cloud import bigquery

bq = bigquery.Client(project=PROJECT_ID, location=REGION)

def run(sql):
    config = bigquery.QueryJobConfig(maximum_bytes_billed=1_000_000_000)   # safety cap: 1 GB per query
    return bq.query(sql, job_config=config).to_dataframe()

pd.set_option("display.max_rows", 200, "display.max_columns", 100, "display.width", 200)
MATCHES = f"{PROJECT_ID}.tennis_clean.matches"

## Create the venue list
Weather is requested by latitude and longitude, but the data only has tournament names. This notebook turns each name into a place and then into coordinates, using Open-Meteo's free geocoding API Names are keyed on the cleaned tournament name.


In [ ]:
venues = run(f"""
SELECT name_key,
       MIN(season) AS first_year, MAX(season) AS last_year,
       COUNT(DISTINCT tourney_id) AS editions,
       ARRAY_TO_STRING(ARRAY_AGG(DISTINCT tourney_level), ',') AS levels
FROM `{MATCHES}`
GROUP BY name_key ORDER BY name_key
""")

# Name in the data -> the place to search for
SEARCH = {
    "australian open": "Melbourne", "melbourne 1": "Melbourne", "melbourne 2": "Melbourne",
    "gippsland trophy": "Melbourne", "yarra valley classic": "Melbourne",
    "grampians trophy": "Melbourne", "phillip island trophy": "Melbourne",
    "roland garros": "Paris", "us open": "New York", "wimbledon": "London",
    "queen's club": "London", "japan open tokyo": "Tokyo",
    "'s hertogenbosch": "'s-Hertogenbosch", "s hertogenbosch": "'s-Hertogenbosch",
    "s-hertogenbosch": "'s-Hertogenbosch", "nurnberg": "Nuremberg",
    "cluj napoca": "Cluj-Napoca", "cluj-napoca 1": "Cluj-Napoca", "cluj-napoca 2": "Cluj-Napoca",
    "adelaide 1": "Adelaide", "adelaide 2": "Adelaide", "charleston 1": "Charleston",
    "guadalajara 2": "Guadalajara", "chicago 2": "Chicago", "hua hin 2": "Hua Hin",
    "calvia": "Calvia", "st. petersburg": "Saint Petersburg", "nur-sultan": "Astana",
}
venues["search_name"] = venues["name_key"].map(SEARCH).fillna(venues["name_key"].str.title())

names = sorted(venues["search_name"].unique())
print(len(venues), "tournament names ->", len(names), "places to geocode")

## Reviewed corrections
The first geocoding results were reviewed by hand. For these places the top result was wrong, so the query or the chosen result is set here:
- `REFETCH`: search for different text, or restrict to a country.
- `PICK_INDEX`: use a result other than the first.

In [ ]:
REFETCH = {                                   # search_name -> (text to search, country code or None)
    "Bol": ("Bol", "HR"),
    "Granby": ("Granby", "CA"),
    "Marrakech": ("Marrakesh", None),
    "Quebec City": ("Quebec", "CA"),
    "Stanford": ("Palo Alto", "US"),
    "Mallorca": ("Calvia", None),
}
PICK_INDEX = {"Indian Wells": 1}              # top result is Indio; the town itself is the second result

queries = {n: REFETCH.get(n, (n, None)) for n in names}

## Fetch and save the raw responses
All responses are kept in one compressed JSON Lines file (one line per query) instead of one small file per query.
Responses already stored (in a packed file, or as single files from earlier runs) are reused, so re-running does not call the API again for anything already stored.

The API's  output is in `response`, and `request`.`fetched_at_utc` records how and when it was obtained.

In [ ]:
import gzip, hashlib, json, re, time
import requests
from datetime import datetime, timezone
from google.cloud import storage

gcs = storage.Client(project=PROJECT_ID)
bucket = gcs.bucket(BUCKET_NAME)

GEO_URL = "https://geocoding-api.open-meteo.com/v1/search"
GEO_PREFIX = "raw/open_meteo/geocoding"
PACKED_PREFIX = f"{GEO_PREFIX}/packed/"
used_keys = set()      # the records this run actually uses (recorded in the manifest)

def slug(text):
    return re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")

# Responses already stored: single files from earlier runs first, then packed files (newer ones win)
stored = {}
for b in gcs.list_blobs(BUCKET_NAME, prefix=f"{GEO_PREFIX}/"):
    if b.name.endswith(".json"):
        stored[b.name.rsplit("/", 1)[1][:-len(".json")]] = json.loads(b.download_as_text())
for b in sorted(gcs.list_blobs(BUCKET_NAME, prefix=PACKED_PREFIX), key=lambda b: b.time_created):
    for line in gzip.decompress(b.download_as_bytes()).splitlines():
        rec = json.loads(line)
        stored[rec.pop("key")] = rec
print("responses already stored:", len(stored))

def geocode(name, country=None):
    """Return the stored record for this query, fetching it only if it is new."""
    key = slug(name) + (f"__{country}" if country else "")
    used_keys.add(key)
    if key in stored:
        return stored[key]

    params = {"name": name, "count": 5, "language": "en", "format": "json"}
    if country:
        params["countryCode"] = country
    r = requests.get(GEO_URL, params=params, timeout=30)
    r.raise_for_status()

    record = {
        "request": {"url": GEO_URL, "params": params},
        "fetched_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "response": r.json(),          # exactly as returned by the API
    }
    stored[key] = record
    time.sleep(0.2)                    # be polite to the free API
    return record

records = {n: geocode(*queries[n]) for n in names}
print("Responses used:", len(records))

# Pack the records this run uses into one file. gzip with mtime=0, so the same records always give the same bytes.
# The file name holds the checksum, so a file is never overwritten and an unchanged set is not stored twice.
lines = [json.dumps({"key": k, **stored[k]}, sort_keys=True) for k in sorted(used_keys)]
packed = gzip.compress(("\n".join(lines) + "\n").encode(), mtime=0)
packed_sha = hashlib.sha256(packed).hexdigest()
GEO_FILE = f"{PACKED_PREFIX}geocoding_{packed_sha[:12]}.jsonl.gz"
blob = bucket.blob(GEO_FILE)
if blob.exists():
    print(GEO_FILE, ": already stored, identical")
else:
    blob.upload_from_string(packed, content_type="application/gzip", if_generation_match=0)
    print(GEO_FILE, ": saved |", len(lines), "records |", len(packed), "bytes")

## Choose the result for each place

In [ ]:
rows = []
for name in names:
    text, country = queries[name]
    results = records[name]["response"].get("results", [])
    idx = PICK_INDEX.get(name, 0)
    assert len(results) > idx, f"No usable result for {name}"
    r = results[idx]
    if name in REFETCH:
        decision = f"re-searched as '{text}'" + (f" in {country}" if country else "")
    elif name in PICK_INDEX:
        decision = f"result #{idx + 1} chosen by hand"
    else:
        decision = "top result"
    rows.append({
        "search_name": name, "place": r["name"], "region": r.get("admin1"),
        "country_code": r.get("country_code"), "latitude": r["latitude"],
        "longitude": r["longitude"], "elevation_m": r.get("elevation"),
        "timezone": r.get("timezone"), "geonames_id": r.get("id"),
        "population": r.get("population"), "decision": decision,
    })

locations = pd.DataFrame(rows)
locations[["search_name", "place", "region", "country_code", "population", "decision"]]

## Checks
Check the results are sensible (yes)

In [ ]:
print("places:", len(locations))
print("duplicate search names:", locations["search_name"].duplicated().sum())
print("missing coordinates:", locations["latitude"].isna().sum(),
      "| missing timezone:", locations["timezone"].isna().sum())
assert set(venues["search_name"]) <= set(locations["search_name"]), "Some venues have no location"
print(locations["country_code"].value_counts().head(8))
locations.sort_values("elevation_m", ascending=False).head(5)[["place", "country_code", "elevation_m"]]

## Save the tables
Two tables in BigQuery (the name-to-place map and the places with coordinates). BigQuery is their one home: they are no longer also copied as CSV files to the bucket. The manifest below records each table's row count and content fingerprint, so any change is detected and versioned.

In [ ]:
name_map = venues[["name_key", "search_name"]]

for table, df in [("venue_name_map", name_map), ("venue_locations", locations)]:
    bq.load_table_from_dataframe(
        df, f"{PROJECT_ID}.tennis_clean.{table}",
        job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")).result()
    print(table, "rows:", bq.get_table(f"{PROJECT_ID}.tennis_clean.{table}").num_rows)

## Manifest of the geocoding file and the tables this run produced

In [ ]:
geo_entry = {"file": GEO_FILE, "bytes": len(packed), "sha256": packed_sha,
             "records": len(lines), "keys": sorted(used_keys)}
tables = [table_version(bq, f"{PROJECT_ID}.tennis_clean.{t}") for t in ["venue_name_map", "venue_locations"]]

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
manifest_path = f"manifests/open_meteo_geocoding_{stamp}.json"
bucket.blob(manifest_path).upload_from_string(
    json.dumps({"endpoint": GEO_URL, "files": [geo_entry], "tables": tables,
                "reviewed_corrections": {"REFETCH": REFETCH, "PICK_INDEX": PICK_INDEX}, **RUN_INFO}, indent=2),
    content_type="application/json")
print(len(lines), "records recorded in", manifest_path)

matches_version = table_version(bq, MATCHES)
register(bucket, "raw/open_meteo_geocoding", packed_sha[:12], outputs=[geo_entry],
         parents=[GEO_URL, matches_version], manifest_path=manifest_path)
for t in tables:
    register(bucket, t["table"], t["fingerprint"], outputs=[t],
             parents=[{"file": GEO_FILE, "sha256": packed_sha}, matches_version], manifest_path=manifest_path)

#Compare every single file with its packed record

In [ ]:
packed = {}
for line in gzip.decompress(bucket.blob(GEO_FILE).download_as_bytes()).splitlines():
    rec = json.loads(line)
    packed[rec.pop("key")] = rec
singles = [b for b in gcs.list_blobs(BUCKET_NAME, prefix=f"{GEO_PREFIX}/") if b.name.endswith(".json")]
diff = [b.name for b in singles
        if json.loads(b.download_as_text()) != packed.get(b.name.rsplit("/", 1)[1][:-len(".json")])]
print("single files:", len(singles), "| packed records:", len(packed), "| differing:", len(diff))